In [1]:
from mpramnist.Gosai2024.dataset import GosaiDataset

from mpramnist.Lee2025.dataset import LeeDataset
from mpramnist.Lee2025.trainer import LitModel_Lee

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L
from lightning.pytorch.callbacks import ModelCheckpoint

from torchmetrics import PearsonCorrCoef

import pandas as pd

BATCH_SIZE = 1024
NUM_WORKERS = 8

In [2]:
import warnings
warnings.filterwarnings("ignore")


# Current Workflow

In this notebook, we:

1. Train the **MPRALegNet** model on the **Gosai SK-N-SH dataset**

2. Assess its predictive power using **Lee's SNPs**

## Custom Gosai 

In [3]:
cell_types = ["SKNSH"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_own = GosaiDataset(
    split="train",
    transform=train_transform,
    filtration="own",
    cell_types=cell_types,
    stderr_columns=["SKNSH_lfcSE"],  # change as you want
    stderr_threshold=1.0,  # change as you want
    std_multiple_cut=6.0,  # change as you want
    up_cutoff_move=3.0,  # change as you want
    duplication_cutoff=0.5,  # change as you want
    root="/media/storage/lizzzafomenko/data",
)
# Use the same parameters to valid and test
val_dataset_own = GosaiDataset(split="val", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='/media/storage/lizzzafomenko/data',)
test_dataset_own = GosaiDataset(split="test", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='/media/storage/lizzzafomenko/data',)

print(train_dataset_own)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 842024
    Root location: /media/storage/lizzzafomenko/data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Gosai dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


In [4]:
# encapsulate data into DataLoader form
train_loader = DataLoader(dataset=train_dataset_own, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset_own[0][0])
out_channels = len(cell_types)

In [5]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Lee(
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
    use_one_cycle = True
)

In [6]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)

trainer = L.Trainer(
    accelerator="gpu",
    devices=[3],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
    callbacks=[checkpoint_callback],
)

trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | tra

Sanity Checking: |          | 0/? [00:00<?, ?it/s]


----------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 1.64248 | Val Pearson: -0.05681 | Train Pearson: nan 
----------------------------------------------------------------------------



Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...


-------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 0.56968 | Val Pearson: 0.84908 | Train Pearson: 0.73036 
-------------------------------------------------------------------------------



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.4692828953266144
      test_pearson          0.7975703477859497
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.4692828953266144, 'test_pearson': 0.7975703477859497}]

In [7]:
best_model_path = checkpoint_callback.best_model_path
seq_model = LitModel_Lee.load_from_checkpoint(best_model_path, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1, use_one_cycle = True,)

In [8]:
forw_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])
revcomp_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(1), t.Seq2Tensor()])

In [9]:
predict_forward_dataset = LeeDataset(split = 'test', length = 150, transform=forw_transform, root = '/media/storage/lizzzafomenko/data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = LeeDataset(split = 'test', length = 150,  transform=revcomp_transform, root = '/media/storage/lizzzafomenko/data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [10]:
forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

**emVar** - Variants that exhibit significant differential regulatory activity between risk and protective alleles (FDR<0.05) and are located in MPRA-active element

**daVar** - Variants that exhibit significant differential regulatory activity between risk and protective alleles (FDR<0.05)

**all** - All variants from the experiment

In [16]:
MAP_VARIANTS_TYPE = {'emVar': [1], 'daVar': [1, 2], 'all': [1, 2, 3, 4]}

def get_variant_predictions(forw_preds, revcomp_preds, variant_types = ['emVar', 'daVar', 'all']):

    targets = torch.cat([pred["target"] for pred in forw_preds])
    variant_type = torch.cat([pred["variant_type"] for pred in forw_preds])
    prediction_sign = torch.cat([pred["reverse_prediction"] for pred in forw_preds])

    y_preds_forw_ref = torch.cat([pred["ref_predicted"] for pred in forw_preds])
    y_preds_forw_alt = torch.cat([pred["alt_predicted"] for pred in forw_preds])


    y_preds_revcomp_ref = torch.cat([pred["ref_predicted"] for pred in revcomp_preds])
    y_preds_revcomp_alt = torch.cat([pred["alt_predicted"] for pred in revcomp_preds])

    y_preds_ref = torch.mean(torch.stack([y_preds_forw_ref, y_preds_revcomp_ref]), dim=0)
    y_preds_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_revcomp_alt]), dim=0)

    variant_prediction = (y_preds_alt - y_preds_ref) * prediction_sign
    
    pears = PearsonCorrCoef()

    for i in range(len(variant_types)):

        mask = torch.isin(variant_type, torch.tensor(MAP_VARIANTS_TYPE[variant_types[i]]))
        pearsonr = pears(variant_prediction.squeeze()[mask], targets.squeeze()[mask])

        print(f'{variant_types[i]} (n = {mask.sum().item()}):   {pearsonr:.6f}')



In [18]:
get_variant_predictions(forw_preds, revcomp_preds, ['emVar', 'daVar', 'all'])

emVar (n = 683):   0.013937
daVar (n = 4472):   0.007249
all (n = 13261):   0.003717
